# ==============================================================================
# 📺 MARKETING MULTI-CHANNEL ATTRIBUTION: ADVERTISING SALES VIA STACKING REGRESSOR
# ==============================================================================
### Core Theoretical Principles:
Stacking Regression (Breiman, 1996) trains a Meta-Learner (Level-1) on the Out-Of-Fold (OOF) cross-validated predictions of Level-0 base regressors:
$$\tilde{X}_i = [\hat{y}_{1,\text{OOF}}(x_i), \dots, \hat{y}_{M,\text{OOF}}(x_i)]$$
$$\hat{y} = g(\tilde{X})$$

By training a regularized Ridge Meta-Learner, the ensemble finds the mathematically optimal linear combination of individual models without overfitting.


In [1]:
# STEP 1: ENTERPRISE ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Marketing Multi-Channel Environment ready.")


✅ STEP 1: Marketing Multi-Channel Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET HARMONIZATION
Loading Advertising dataset (200 sales records across TV, Radio, Newspaper).


In [2]:
# STEP 2 & 3: LOAD & HARMONIZE
data_path = 'data/advertising/advertising.csv'
df = pd.read_csv(data_path)

if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

df = df.rename(columns={
    'TV Ad Budget ($)': 'tv',
    'Radio Ad Budget ($)': 'radio',
    'Newspaper Ad Budget ($)': 'newspaper',
    'Sales ($)': 'sales'
})
df.columns = [c.strip().lower().split(' ')[0] for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]} campaigns, {df.shape[1]} attributes")
print(f"Sales Target Summary:\n{df['sales'].describe().round(2)}")
df.head(3)


📊 Dataset Shape: 200 campaigns, 4 attributes
Sales Target Summary:
count    200.00
mean      14.02
std        5.22
min        1.60
25%       10.38
50%       12.90
75%       17.40
max       27.00
Name: sales, dtype: float64


,tv,radio,newspaper,sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,9.3


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and continuous target `sales`.


In [3]:
# STEP 4 & 5: SEGREGATION
X = df.drop(columns=['sales'])
y = df['sales']

print(f"Marketing Channels: {X.columns.tolist()}")
print(f"Missing attributes: {X.isnull().sum().sum()}")


Marketing Channels: ['tv', 'radio', 'newspaper']
Missing attributes: 0


In [4]:
# STEP 6: TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train Campaigns: {X_train.shape[0]}")
print(f"Test Campaigns : {X_test.shape[0]}")


Train Campaigns: 160
Test Campaigns : 40


## ⚠️ STEP 7 & 8: BASELINE STACKING REGRESSOR FIT
Stacking OLS, Decision Tree, and KNN with a Ridge Meta-Learner.


In [5]:
# STEP 7 & 8: BASELINE STACKING FIT
pipe_ols = Pipeline([('scaler', StandardScaler()), ('ols', LinearRegression())])
pipe_dt = Pipeline([('dt', DecisionTreeRegressor(max_depth=4, random_state=42))])
pipe_knn = Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsRegressor(n_neighbors=5))])

stack_reg = StackingRegressor(
    estimators=[('ols', pipe_ols), ('dt', pipe_dt), ('knn', pipe_knn)],
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1
)
stack_reg.fit(X_train, y_train)

r2_stack = r2_score(y_test, stack_reg.predict(X_test))
mae_stack = mean_absolute_error(y_test, stack_reg.predict(X_test))

print("="*65)
print(f"Stacked Meta-Regression R² : {r2_stack * 100:.2f}% | MAE: ${mae_stack:.2f}k")
print("="*65)


Stacked Meta-Regression R² : 95.43% | MAE: $0.96k


## 🎯 STEP 9: TUNING META-LEARNER VIA 5-FOLD CV
Optimizing Ridge meta-learner penalty `final_estimator__alpha` and `passthrough`.


In [6]:
# STEP 9: GRID SEARCH
param_grid = {
    'final_estimator__alpha': [0.1, 1.0, 10.0],
    'passthrough': [False, True]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(stack_reg, param_grid=param_grid, cv=cv, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Meta-Learner Parameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV R²: {grid.best_score_*100:.2f}%")


🏆 Best Meta-Learner Parameters: {'final_estimator__alpha': 10.0, 'passthrough': True}
🎯 Best 5-Fold CV R²: 94.10%


## 📊 STEP 10 & 11: CAMPAIGN REVENUE REPORT & RESIDUALS
Evaluating revenue forecast precision.


In [7]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("📋 MARKETING CAMPAIGN FORECAST METRICS:")
print(f"   Mean Absolute Error (MAE) : ${mae:.2f}k")
print(f"   Root Mean Squared Error   : ${rmse:.2f}k")
print(f"   R² Score                  : {r2 * 100:.2f}%")


📋 MARKETING CAMPAIGN FORECAST METRICS:
   Mean Absolute Error (MAE) : $0.94k
   Root Mean Squared Error   : $1.16k
   R² Score                  : 95.70%


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & CAMPAIGN ALLOCATION SMOKE TEST
Deploying pipeline and validating marketing allocation budget optimization latency.


In [8]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/advertising_stacking_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_campaign = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_sales = loaded_pipe.predict(sample_campaign)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n📈 LIVE MARKETING SPEND FORECAST:")
print(f"   Forecast Campaign Sales : 📦 ${pred_sales:,.2f}k")
print(f"   Actual Realized Sales   : ${y_test.iloc[0]:,.2f}k")
print(f"   Forecast Latency        : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/advertising_stacking_pipeline.joblib (8,003 bytes)

📈 LIVE MARKETING SPEND FORECAST:
   Forecast Campaign Sales : 📦 $18.32k
   Actual Realized Sales   : $16.90k
   Forecast Latency        : 7.162 ms (SLA < 2.0ms: CHECK)
